# ✉️ Messages
  <img src="./assets/LC_Messages.png" width="500">

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

## Setup

Load and/or check for needed environmental variables

In [1]:
import os
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

AI_API_KEY=****1zgM
AI_MODEL=google_genai:gemini-3.5-flash-lite
LANGSMITH_API_KEY=****0a4f
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=My Test Project


## Human👨‍💻 and AI 🤖 Messages

In [2]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model=os.getenv("AI_MODEL", "google_genai:gemini-3.5-flash-lite"),
    system_prompt="Ты универсальный комик"
)

In [3]:
human_msg = HumanMessage("Привет, как ты??")

result = agent.invoke({"messages": [human_msg]})

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [28]:
import textwrap
from typing import Any
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage


def _extract_text(content: Any) -> str:
    """Извлекает чистый текст из message.content (str или list[dict])."""

    try:
        if isinstance(content, str):
            return content.strip()

        if isinstance(content, list):
            text_parts = []
            for block in content:
                if isinstance(block, dict):
                    if block.get("type","") == "text":
                        text_parts.append(block.get("text", ""))
                elif isinstance(block, str):
                    text_parts.append(block)
            return "\n".join(text_parts).strip()

        return str(content).strip()
    except Exception as e:
        return f"Error extracting text: {e}"


def print_result(loc_result) -> None:
    """Красиво печатает пару сообщений (Human и AI) с поддержкой любого типа content."""
    loc_messages = loc_result.get("messages", [])

    if isinstance(loc_messages, list) and len(loc_messages)>1:
        human_text = _extract_text(loc_messages[0].content)
        ai_raw_text = _extract_text(loc_messages[1].content)
    else:
        human_text = f"loc_result['messages'] is not a list with at least two elements ({type(loc_result)})"
        ai_raw_text = human_text

    # Добавляем отступ в 4 пробела для всех строк ответа AI, начиная со второй
    ai_lines = ai_raw_text.splitlines()
    if ai_lines:
        first_line = ai_lines[0]
        rest = "\n".join(ai_lines[1:])
        ai_text = f"{first_line}\n{textwrap.indent(rest, ' ' * 4)}" if rest else first_line
    else:
        ai_text = ""

    print(f"Human: {human_text}\n")
    print(f"AI: {ai_text}\n")


In [29]:
print_result(result)

Human: Привет, как ты??

AI: Привет! Живем потихоньку, батарейка на 15%, но режим энергосбережения творит чудеса. 😂 

    Сам как? Какими судьбами на мою скромную стену комедии? Пришел поржать или просто проверить, не захватили ли еще искусственные интеллекты мир? (Спойлер: пока нет, мы еще тупим на капчах с светофорами).



In [6]:
print(type(result["messages"][-1]))

<class 'langchain_core.messages.ai.AIMessage'>


In [7]:
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}\n")

human: Привет, как ты??

ai: [{'type': 'text', 'text': 'Привет! Живем потихоньку, батарейка на 15%, но режим энергосбережения творит чудеса. 😂 \n\nСам как? Какими судьбами на мою скромную стену комедии? Пришел поржать или просто проверить, не захватили ли еще искусственные интеллекты мир? (Спойлер: пока нет, мы еще тупим на капчах с светофорами).', 'extras': {'signature': 'El4KXAFpFH0T8lYh9T7tZ+Q0U4EtWAUz480uZvUjnkLo0pkVpgn9sGaeP4BJWWo/pbOccfkyDF90yxsw/xlw0ejJpf6fq2gzt9s3TQx2HyRooIDy8qL8A9lUgUDRT9v8'}}]



### Altenative formats
#### Strings
There are situations where LangChain can infer the role from the context, and a simple string is enough to create a message. 

In [ ]:
agent = create_agent(
    model=os.getenv("AI_MODEL", "google_genai:gemini-3.5-flash-lite"),
    system_prompt="You are a terse sports poet.",  # This is a SystemMessage under the hood
)

In [ ]:
result = agent.invoke({"messages": "Tell me about baseball"})   # This is a HumanMessage under the hood
print(result["messages"][-1].content)

#### Dictionaries

In [ ]:
result = agent.invoke(
    {"messages": {"role": "user", "content": "Write a haiku about sprinters"}}
)
print(result["messages"][-1].content)

There are multiple roles:
```python
messages = [
    {"role": "system", "content": "You are a sports poetry expert who completes haikus that have been started"},
    {"role": "user", "content": "Write a haiku about sprinters"},
    {"role": "assistant", "content": "Feet don't fail me..."}
]
```

## Output Format
### messages
Let's create a tool so agent will create some tool messages. 

In [ ]:
from langchain_core.tools import tool

@tool
def check_haiku_lines(text: str):
    """Check if the given haiku text has exactly 3 lines.

    Returns None if it's correct, otherwise an error message.
    """
    # Split the text into lines, ignoring leading/trailing spaces
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    print(f"checking haiku, it has {len(lines)} lines:\n {text}")

    if len(lines) != 3:
        return f"Incorrect! This haiku has {len(lines)} lines. A haiku must have exactly 3 lines."
    return "Correct, this haiku has 3 lines."

In [ ]:
agent = create_agent(
    model=os.getenv("AI_MODEL", "google_genai:gemini-3.5-flash-lite"),
    tools=[check_haiku_lines],
    system_prompt="You are a sports poet who only writes Haiku. You always check your work.",
)

In [ ]:
result = agent.invoke({"messages": "Please write me a poem"})

In [ ]:
result["messages"][-1].content

In [ ]:
print(len(result["messages"]))

In [ ]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

### Other useful information
Above, the print messages have just been selecting pieces of the information stored in the messages list. Let's dig into all the information that is available!

In [ ]:
result

You can select just the last message, and you can see where the final message is coming from.

In [ ]:
result["messages"][-1]

In [ ]:
result["messages"][-1].usage_metadata

In [ ]:
result["messages"][-1].response_metadata

### Try it on your own!
Change the system prompt, use the `pretty_printer` to print some messages or dig through `results` on your own. Notice the Human, AI and Tool messages and some of their associated metadata. Notice how the final results provide a complete history of the agents activity!

In [ ]:
agent = create_agent(
    model=os.getenv("AI_MODEL", "google_genai:gemini-3.5-flash-lite"),
    tools=[check_haiku_lines],
    system_prompt="Your SYSTEM prompt here",
)

In [ ]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()